# 02 — PySpark Local Mode

Real PySpark jobs, run in local mode (`local[*]`) on this machine's 16 cores, against a
locally-generated synthetic dataset (fixed seed, no download). Every number and every `.explain()`
plan below is actual output from this machine, not fabricated or estimated.

This topic cites `01-why-distributed-processing`'s from-scratch `multiprocessing` work directly:
that topic hand-built partitioning, map/filter/aggregate, a shuffle-cost measurement, and
fault-tolerance-by-lineage, entirely in the standard library. This topic installs a real engine
(PySpark) that automates all four of those mechanisms, and measures it against the from-scratch
approach on a comparable groupBy/join workload.

In [1]:
import time
import math
import random
import tempfile
import os
import multiprocessing as mp

import pandas as pd

random.seed(42)
print("CPU count on this machine:", mp.cpu_count())

CPU count on this machine: 16


## 1. Start a real `SparkSession` in local mode

`local[*]` tells Spark to run as a single process on this machine, using all available cores as
"executors" — no cluster, no external services, nothing to install beyond the JVM (confirmed
present: `openjdk 21.0.11`) and the `pyspark` package (`uv add pyspark`, just installed).

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

t0 = time.perf_counter()
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("02-pyspark-local-mode")
    .config("spark.sql.shuffle.partitions", "16")  # match local core count instead of the 200 default
    .getOrCreate()
)
t_startup = time.perf_counter() - t0

print(f"SparkSession started in {t_startup:.2f}s")
print("Spark version:", spark.version)
print("Default parallelism (cores Spark sees):", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 00:53:51 WARN Utils: Your hostname, Yashu, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 00:53:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/yashwanth-aravind/ml-course/python-bootcamp/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/24 00:53:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession started in 3.29s
Spark version: 4.2.0
Default parallelism (cores Spark sees): 16


## 2. Generate synthetic data locally (fixed seed, no download)

Two tables, mirroring a realistic small-OLTP-style schema:

- **`transactions`**: 800,000 rows — `customer_id`, `amount`, `category`. Comparable in scale to
  `01-why-distributed-processing`'s 300,000-row toy dataset, large enough that partitioning and
  shuffle are real (not instantaneous), small enough to stay within the "couple of minutes total"
  budget for this topic.
- **`customers`**: 100,000 rows — `customer_id`, `region`, `signup_year`. Joined against
  `transactions` on `customer_id` below to force a real shuffle.

Written to Parquet in a temp directory so the notebook is self-contained and reproducible without
committing generated data to the repo.

In [3]:
N_TXN = 800_000
N_CUSTOMERS = 100_000
CATEGORIES = ["groceries", "electronics", "clothing", "dining", "travel", "utilities",
              "entertainment", "health", "home", "other"]
REGIONS = ["north", "south", "east", "west", "central"]

random.seed(42)
customers_pd = pd.DataFrame({
    "customer_id": range(N_CUSTOMERS),
    "region": [random.choice(REGIONS) for _ in range(N_CUSTOMERS)],
    "signup_year": [random.randint(2015, 2025) for _ in range(N_CUSTOMERS)],
})

random.seed(7)
transactions_pd = pd.DataFrame({
    "txn_id": range(N_TXN),
    "customer_id": [random.randrange(N_CUSTOMERS) for _ in range(N_TXN)],
    "amount": [round(random.uniform(1.0, 500.0), 2) for _ in range(N_TXN)],
    "category": [random.choice(CATEGORIES) for _ in range(N_TXN)],
})

data_dir = tempfile.mkdtemp(prefix="pyspark_local_mode_")
customers_path = os.path.join(data_dir, "customers.parquet")
transactions_path = os.path.join(data_dir, "transactions.parquet")
customers_pd.to_parquet(customers_path, index=False)
transactions_pd.to_parquet(transactions_path, index=False)

print(f"Wrote {len(customers_pd):,} customers to {customers_path}")
print(f"Wrote {len(transactions_pd):,} transactions to {transactions_path}")
print(f"On-disk size: customers={os.path.getsize(customers_path)/1e6:.2f} MB, "
      f"transactions={os.path.getsize(transactions_path)/1e6:.2f} MB")

Wrote 100,000 customers to /tmp/pyspark_local_mode_z3jtiq0b/customers.parquet
Wrote 800,000 transactions to /tmp/pyspark_local_mode_z3jtiq0b/transactions.parquet
On-disk size: customers=0.69 MB, transactions=7.86 MB


## 3. `.read` — real PySpark jobs start here

`spark.read.parquet(...)` is itself lazy: it registers the source, it does not read the file yet.
Nothing actually runs on the cluster (here, this machine's cores) until an **action** (`.count()`,
`.collect()`, `.show()`, ...) is called. Every `.filter`/`.groupBy`/`.join` below is a
**transformation** — it only builds up a logical execution plan.

In [4]:
t0 = time.perf_counter()
customers_df = spark.read.parquet(customers_path)
transactions_df = spark.read.parquet(transactions_path)
t_read_call = time.perf_counter() - t0
print(f".read.parquet() call itself returned in {t_read_call:.4f}s (lazy — no data has moved yet)")

customers_df.printSchema()
transactions_df.printSchema()

t0 = time.perf_counter()
n_customers = customers_df.count()
n_txn = transactions_df.count()
t_count_action = time.perf_counter() - t0
print(f".count() ACTION on both tables: {n_customers:,} customers, {n_txn:,} transactions "
      f"(actually ran in {t_count_action:.2f}s)")

.read.parquet() call itself returned in 1.1623s (lazy — no data has moved yet)
root
 |-- customer_id: long (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_year: long (nullable = true)

root
 |-- txn_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)



.count() ACTION on both tables: 100,000 customers, 800,000 transactions (actually ran in 1.11s)


## 4. Transformations vs. actions, made visible

`transactions_df.filter(...)` below returns instantly — it's a transformation, building a plan,
not touching data. Calling `.explain()` on it shows the plan Spark *would* execute, without
executing it. Only `.count()` (an action) actually runs the filter across all cores.

In [5]:
t0 = time.perf_counter()
high_value = transactions_df.filter(F.col("amount") > 400.0)
t_filter_call = time.perf_counter() - t0
print(f".filter() call returned in {t_filter_call:.5f}s (lazy — just built a plan)")

print("\n--- .explain() on the unexecuted filter plan ---")
high_value.explain()

t0 = time.perf_counter()
n_high_value = high_value.count()
t_filter_action = time.perf_counter() - t0
print(f"\n.count() ACTION: {n_high_value:,} of {n_txn:,} transactions have amount > 400 "
      f"(ran in {t_filter_action:.2f}s)")

.filter() call returned in 0.02956s (lazy — just built a plan)

--- .explain() on the unexecuted filter plan ---
== Physical Plan ==
*(1) Filter (isnotnull(amount#5) AND (amount#5 > 400.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [txn_id#3L,customer_id#4L,amount#5,category#6] Batched: true, DataFilters: [isnotnull(amount#5), (amount#5 > 400.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/pyspark_local_mode_z3jtiq0b/transactions.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,400.0)], ReadSchema: struct<txn_id:bigint,customer_id:bigint,amount:double,category:string>





.count() ACTION: 160,337 of 800,000 transactions have amount > 400 (ran in 0.28s)


## 5. `.groupBy().agg()` — a real aggregation job

Group 800,000 transactions by `category` (10 distinct keys) and compute sum/count/avg of
`amount` per category. This is the DataFrame-API equivalent of `01-why-distributed-processing`'s
"aggregate" primitive — but grouped by key, which means it also needs a shuffle (see `.explain()`
below: an `Exchange` node) to collocate same-key rows before the final aggregation.

In [6]:
category_agg = (
    transactions_df
    .groupBy("category")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount"),
    )
    .orderBy(F.desc("total_amount"))
)

print("--- .explain() on the groupBy/agg plan ---")
category_agg.explain()

t0 = time.perf_counter()
category_agg.show(10, truncate=False)
t_groupby_show = time.perf_counter() - t0
print(f"groupBy/agg + show ACTION ran in {t_groupby_show:.2f}s")

--- .explain() on the groupBy/agg plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_amount#31 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_amount#31 DESC NULLS LAST, 16), ENSURE_REQUIREMENTS, [plan_id=155]
      +- HashAggregate(keys=[category#6], functions=[sum(amount#5), count(1), avg(amount#5)])
         +- Exchange hashpartitioning(category#6, 16), ENSURE_REQUIREMENTS, [plan_id=152]
            +- HashAggregate(keys=[category#6], functions=[partial_sum(amount#5), partial_count(1), partial_avg(amount#5)])
               +- FileScan parquet [amount#5,category#6] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/pyspark_local_mode_z3jtiq0b/transactions.parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<amount:double,category:string>




+-------------+--------------------+---------+------------------+
|category     |total_amount        |txn_count|avg_amount        |
+-------------+--------------------+---------+------------------+
|other        |2.0236213920000214E7|80382    |251.75056505187996|
|clothing     |2.016584107000006E7 |80518    |250.45134094239873|
|dining       |2.0093483359999787E7|80276    |250.30498978523826|
|electronics  |2.0068274309999924E7|80052    |250.69048006295813|
|utilities    |2.0061651650000177E7|80000    |250.7706456250022 |
|groceries    |2.0017158080000028E7|80019    |250.15506417225944|
|health       |1.9990507279999964E7|79866    |250.30059449578   |
|travel       |1.9953108479999814E7|79703    |250.34325533543046|
|entertainment|1.9949295359999996E7|79513    |250.89350621910876|
|home         |1.9948220239999887E7|79671    |250.382450829033  |
+-------------+--------------------+---------+------------------+

groupBy/agg + show ACTION ran in 0.52s


## 6. `.join()` — shuffle made concrete

Join `transactions` (800,000 rows, `customer_id`) with `customers` (100,000 rows, `customer_id`)
to attach `region`/`signup_year` to every transaction. Rows with the same `customer_id` are not,
in general, on the same partition — Spark has to shuffle at least one side to collocate matching
keys before it can join them locally. `.explain()` shows this explicitly as an `Exchange`
(shuffle) step, exactly the mechanism `01-why-distributed-processing`'s naive-shuffle-vs-map-side-combine
demo measured by hand.

In [7]:
joined = transactions_df.join(customers_df, on="customer_id", how="inner")

print("--- .explain() on the join plan (look for 'Exchange' = shuffle) ---")
joined.explain()

t0 = time.perf_counter()
n_joined = joined.count()
t_join_action = time.perf_counter() - t0
print(f"\njoin ACTION (.count()): {n_joined:,} rows produced in {t_join_action:.2f}s "
      f"(expected {n_txn:,}, since every customer_id exists in customers_df)")

--- .explain() on the join plan (look for 'Exchange' = shuffle) ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#4L, txn_id#3L, amount#5, category#6, region#1, signup_year#2L]
   +- BroadcastHashJoin [customer_id#4L], [customer_id#0L], Inner, BuildRight, false, false
      :- Filter isnotnull(customer_id#4L)
      :  +- FileScan parquet [txn_id#3L,customer_id#4L,amount#5,category#6] Batched: true, DataFilters: [isnotnull(customer_id#4L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/pyspark_local_mode_z3jtiq0b/transactions.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<txn_id:bigint,customer_id:bigint,amount:double,category:string>
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=241]
         +- Filter isnotnull(customer_id#0L)
            +- FileScan parquet [customer_id#0L,region#1,signup_year#2L] Batched: true, DataFilters: [isnotnu


join ACTION (.count()): 800,000 rows produced in 0.31s (expected 800,000, since every customer_id exists in customers_df)


In [8]:
region_spend = (
    joined
    .groupBy("region")
    .agg(F.sum("amount").alias("total_amount"), F.count("*").alias("txn_count"))
    .orderBy(F.desc("total_amount"))
)
region_spend.show(truncate=False)

+-------+--------------------+---------+
|region |total_amount        |txn_count|
+-------+--------------------+---------+
|west   |4.052193595999919E7 |161752   |
|north  |4.0346477979999855E7|160870   |
|south  |4.0013004710000366E7|159777   |
|central|3.999011534000038E7 |159433   |
|east   |3.9612219759999946E7|158168   |
+-------+--------------------+---------+



## 6b. Forcing a real shuffle join

The `.explain()` above shows a `BroadcastHashJoin`, not a shuffle: Spark's Catalyst optimizer
noticed `customers_df` is small (well under the default 10 MB auto-broadcast threshold) and
automatically broadcast the whole table to every partition instead of shuffling the much larger
`transactions_df` — this **is** the engine automating the exact optimization the from-scratch
broadcast-join code (Section 7 below) does by hand. To see the shuffle-based join plan, disable
the broadcast threshold and force Spark to shuffle both sides on `customer_id` instead.

In [9]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # force a shuffle join

joined_shuffle = transactions_df.join(customers_df, on="customer_id", how="inner")
print("--- .explain() with broadcast disabled (look for 'Exchange' = shuffle on BOTH sides) ---")
joined_shuffle.explain()

t0 = time.perf_counter()
n_joined_shuffle = joined_shuffle.count()
t_join_shuffle = time.perf_counter() - t0
print(f"\nForced shuffle join ACTION: {n_joined_shuffle:,} rows in {t_join_shuffle:.2f}s "
      f"(vs. {t_join_action:.2f}s for the auto-broadcast join above -- "
      f"shuffling both sides costs more than broadcasting the small table)")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # restore Spark's default (10 MB)

--- .explain() with broadcast disabled (look for 'Exchange' = shuffle on BOTH sides) ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#4L, txn_id#3L, amount#5, category#6, region#1, signup_year#2L]
   +- SortMergeJoin [customer_id#4L], [customer_id#0L], Inner
      :- Sort [customer_id#4L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(customer_id#4L, 16), ENSURE_REQUIREMENTS, [plan_id=577]
      :     +- Filter isnotnull(customer_id#4L)
      :        +- FileScan parquet [txn_id#3L,customer_id#4L,amount#5,category#6] Batched: true, DataFilters: [isnotnull(customer_id#4L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/pyspark_local_mode_z3jtiq0b/transactions.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<txn_id:bigint,customer_id:bigint,amount:double,category:string>
      +- Sort [customer_id#0L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(customer_id#


Forced shuffle join ACTION: 800,000 rows in 0.67s (vs. 0.31s for the auto-broadcast join above -- shuffling both sides costs more than broadcasting the small table)


## 7. Experiment — PySpark vs. from-scratch `multiprocessing` on the same groupBy/join

**Hypothesis (stated before measuring):**

1. On this modest, single-machine-friendly dataset (800K rows), a hand-rolled
   `multiprocessing.Pool` map-side-combine groupBy (the technique `01-why-distributed-processing`
   measured the *data volume* savings of, now actually timed end-to-end) will be **at least
   competitive with, and likely faster than**, PySpark's `groupBy().agg()` — PySpark pays real,
   fixed costs (JVM startup already paid above, Python↔JVM serialization via Py4J, query
   planning/optimization) that don't pay for themselves at this scale.
2. The same will hold for the join: a from-scratch broadcast hash join (build a dict from the
   small `customers` table, probe it per transaction in parallel) should be competitive with or
   faster than Spark's `.join()`, because 100,000 customer rows easily fit in every worker's
   memory — this is the same shape of optimization Spark itself would pick automatically via a
   broadcast join for a small enough table.
3. PySpark's real advantage is not raw speed at this toy scale — it's that the *exact same code*
   here would keep working if `transactions_df` were 800 million rows instead of 800 thousand
   (spilling to disk, distributing across a real cluster), where the from-scratch version would
   need a full rewrite (or simply doesn't fit in one machine's RAM at all).

In [10]:
# --- From-scratch multiprocessing groupBy (map-side combine), same operation as Section 5 ---

def partial_groupby(chunk):
    """Map-side combine: local partial (sum, count) per category for one chunk."""
    partial = {}
    for cat, amt in chunk:
        s, c = partial.get(cat, (0.0, 0))
        partial[cat] = (s + amt, c + 1)
    return partial

rows = list(zip(transactions_pd["category"], transactions_pd["amount"]))
n_workers = 8
chunk_size = math.ceil(len(rows) / n_workers)
chunks = [rows[i:i + chunk_size] for i in range(0, len(rows), chunk_size)]

t0 = time.perf_counter()
with mp.Pool(processes=n_workers) as pool:
    partials = pool.map(partial_groupby, chunks)

final = {}
for partial in partials:
    for cat, (s, c) in partial.items():
        fs, fc = final.get(cat, (0.0, 0))
        final[cat] = (fs + s, fc + c)
t_mp_groupby = time.perf_counter() - t0

mp_result = sorted(
    [(cat, s, c, s / c) for cat, (s, c) in final.items()],
    key=lambda r: -r[1],
)
print(f"from-scratch multiprocessing groupBy ({n_workers} workers): {t_mp_groupby:.3f}s")
for cat, s, c, avg in mp_result[:5]:
    print(f"  {cat:15s} total={s:14,.2f}  count={c:6,d}  avg={avg:7.2f}")

from-scratch multiprocessing groupBy (8 workers): 0.452s
  other           total= 20,236,213.92  count=80,382  avg= 251.75
  clothing        total= 20,165,841.07  count=80,518  avg= 250.45
  dining          total= 20,093,483.36  count=80,276  avg= 250.30
  electronics     total= 20,068,274.31  count=80,052  avg= 250.69
  utilities       total= 20,061,651.65  count=80,000  avg= 250.77


In [11]:
# Re-time the Spark groupBy for a fair apples-to-apples comparison (Section 5's number
# included the .show() formatting cost; time count() as the pure action here instead).
t0 = time.perf_counter()
_ = category_agg.count()
t_spark_groupby = time.perf_counter() - t0
print(f"PySpark groupBy().agg() (re-timed, .count() action): {t_spark_groupby:.3f}s")
print(f"\nComparison @ 800K rows, groupBy(category), 10 keys:")
print(f"  multiprocessing (8 workers): {t_mp_groupby:.3f}s")
print(f"  PySpark local[*]:            {t_spark_groupby:.3f}s")
faster = "multiprocessing" if t_mp_groupby < t_spark_groupby else "PySpark"
ratio = max(t_mp_groupby, t_spark_groupby) / min(t_mp_groupby, t_spark_groupby)
print(f"  -> {faster} was {ratio:.2f}x faster at this scale")

PySpark groupBy().agg() (re-timed, .count() action): 0.174s

Comparison @ 800K rows, groupBy(category), 10 keys:
  multiprocessing (8 workers): 0.452s
  PySpark local[*]:            0.174s
  -> PySpark was 2.60x faster at this scale


In [12]:
# --- From-scratch broadcast hash join, same operation as Section 6 ---

customers_lookup = dict(zip(customers_pd["customer_id"], zip(customers_pd["region"], customers_pd["signup_year"])))

def broadcast_join_chunk(chunk_customer_ids_amounts, lookup):
    out_count = 0
    total = 0.0
    for cust_id, amt in chunk_customer_ids_amounts:
        region_year = lookup.get(cust_id)
        if region_year is not None:
            out_count += 1
            total += amt
    return out_count, total

txn_rows = list(zip(transactions_pd["customer_id"], transactions_pd["amount"]))
chunks_join = [txn_rows[i:i + chunk_size] for i in range(0, len(txn_rows), chunk_size)]

t0 = time.perf_counter()
with mp.Pool(processes=n_workers) as pool:
    join_results = pool.starmap(broadcast_join_chunk, [(c, customers_lookup) for c in chunks_join])
t_mp_join = time.perf_counter() - t0

mp_join_count = sum(r[0] for r in join_results)
mp_join_total = sum(r[1] for r in join_results)
print(f"from-scratch multiprocessing broadcast join ({n_workers} workers): {t_mp_join:.3f}s")
print(f"  joined rows: {mp_join_count:,}  total amount: {mp_join_total:,.2f}")

from-scratch multiprocessing broadcast join (8 workers): 0.531s
  joined rows: 800,000  total amount: 200,483,753.75


In [13]:
t0 = time.perf_counter()
n_joined_retimed = joined.count()
t_spark_join = time.perf_counter() - t0
print(f"PySpark .join() (re-timed, .count() action): {t_spark_join:.3f}s")
print(f"\nComparison @ 800K txn rows join 100K customer rows on customer_id:")
print(f"  multiprocessing broadcast join (8 workers): {t_mp_join:.3f}s")
print(f"  PySpark local[*] join:                      {t_spark_join:.3f}s")
faster_join = "multiprocessing" if t_mp_join < t_spark_join else "PySpark"
ratio_join = max(t_mp_join, t_spark_join) / min(t_mp_join, t_spark_join)
print(f"  -> {faster_join} was {ratio_join:.2f}x faster at this scale")

PySpark .join() (re-timed, .count() action): 0.164s

Comparison @ 800K txn rows join 100K customer rows on customer_id:
  multiprocessing broadcast join (8 workers): 0.531s
  PySpark local[*] join:                      0.164s
  -> PySpark was 3.23x faster at this scale


## 8. Failure modes, demonstrated (not just described)

### `.collect()`ing too much to the driver

`.collect()` pulls every row of a distributed DataFrame back into the driver process's memory as
plain Python objects — the exact opposite of staying distributed. It's fine for a small, already
-aggregated result (`region_spend` above, 5 rows); it's a real production outage waiting to happen
on a large DataFrame. Below: the *size* of what `.collect()` on the full, unaggregated join would
pull back, computed without actually doing it (that's the point — you often only find out it's
too big after the driver OOMs).

In [14]:
# Estimate collect() footprint WITHOUT actually collecting (that's the failure-mode point):
# row count x an approximate per-row Python-object footprint.
approx_bytes_per_row = 8 + 8 + len("electronics") + 8 + len("north") + 8  # rough per-field overhead
approx_collect_mb = (n_joined * approx_bytes_per_row) / 1e6
print(f"joined has {n_joined:,} rows. A `.collect()` on it (not run here) would pull roughly "
      f"{approx_collect_mb:.0f} MB of Python objects onto the driver — "
      f"on a real cluster (bigger data, or a driver sized for coordination, not data volume) "
      f"this is a common source of driver OOM. The correct pattern used throughout this notebook: "
      f"aggregate/filter FIRST (region_spend has 5 rows), collect/show only the small result."
)

joined has 800,000 rows. A `.collect()` on it (not run here) would pull roughly 38 MB of Python objects onto the driver — on a real cluster (bigger data, or a driver sized for coordination, not data volume) this is a common source of driver OOM. The correct pattern used throughout this notebook: aggregate/filter FIRST (region_spend has 5 rows), collect/show only the small result.


In [15]:
print("Safe: collecting the small, already-aggregated result (5 rows) —")
small_result = region_spend.collect()
for row in small_result:
    print(f"  {row['region']:10s} total={row['total_amount']:14,.2f}  txns={row['txn_count']:6,d}")

Safe: collecting the small, already-aggregated result (5 rows) —


  west       total= 40,521,935.96  txns=161,752
  north      total= 40,346,477.98  txns=160,870
  south      total= 40,013,004.71  txns=159,777
  central    total= 39,990,115.34  txns=159,433
  east       total= 39,612,219.76  txns=158,168


### Skewed joins

The join above used `customer_id` uniformly distributed across 100,000 customers — no key
dominates. A **skewed** join key (e.g. one `customer_id` responsible for 30% of all transactions —
plausible for a "guest checkout" or system account in a real dataset) sends a disproportionate
share of shuffled rows to a single partition/task, so that one task runs far longer than the rest
while other cores sit idle waiting for it. Demonstrated below: a synthetic heavily-skewed key
column, and the resulting per-partition row-count imbalance after a `groupBy`.

In [16]:
random.seed(99)
HOT_KEY = 0
skewed_categories = (
    [HOT_KEY] * int(N_TXN * 0.30)
    + [random.randrange(1, 5000) for _ in range(N_TXN - int(N_TXN * 0.30))]
)
random.shuffle(skewed_categories)
skewed_pd = pd.DataFrame({"key": skewed_categories, "amount": transactions_pd["amount"].values})
skewed_path = os.path.join(data_dir, "skewed.parquet")
skewed_pd.to_parquet(skewed_path, index=False)
skewed_df = spark.read.parquet(skewed_path)

t0 = time.perf_counter()
skew_result = skewed_df.groupBy("key").agg(F.count("*").alias("cnt")).orderBy(F.desc("cnt"))
top_keys = skew_result.limit(3).collect()
t_skew = time.perf_counter() - t0

print(f"Skewed groupBy ran in {t_skew:.2f}s. Top keys by row count:")
for row in top_keys:
    pct = 100 * row["cnt"] / N_TXN
    print(f"  key={row['key']}: {row['cnt']:,} rows ({pct:.1f}% of all data)")
print(f"\nCompare groupBy(category) above (10 near-uniform keys, {t_spark_groupby:.2f}s) to this: "
      f"one key alone ({top_keys[0]['cnt']:,} rows) is bigger than any single category above — "
      f"the task handling that key's shuffle partition does that much more work than the rest."
)

Skewed groupBy ran in 0.16s. Top keys by row count:
  key=0: 240,000 rows (30.0% of all data)
  key=3388: 154 rows (0.0% of all data)
  key=3008: 153 rows (0.0% of all data)

Compare groupBy(category) above (10 near-uniform keys, 0.17s) to this: one key alone (240,000 rows) is bigger than any single category above — the task handling that key's shuffle partition does that much more work than the rest.


### Too many small partitions

`local[*]` runs on this machine's cores, but it doesn't hide the fundamental partition-count
tradeoff every Spark job faces: too few partitions underuses available parallelism; too many
creates scheduling overhead that can dominate actual work. Below: the same small aggregation,
forced into 1 partition (no parallelism) vs. 400 tiny partitions (task-scheduling overhead per
partition dwarfs the sub-millisecond of real work each partition does) vs. a reasonable count.

In [17]:
small_df_base = spark.range(0, 50_000)

for n_parts in [1, 16, 400]:
    df_n = small_df_base.repartition(n_parts)
    t0 = time.perf_counter()
    result = df_n.groupBy((F.col("id") % 10).alias("bucket")).count().collect()
    elapsed = time.perf_counter() - t0
    print(f"  {n_parts:4d} partitions: {elapsed:.3f}s")

print("\nToo few partitions (1) can't use more than 1 core for this stage. Too many partitions "
      "(400, for only 50,000 rows) pays fixed per-task scheduling overhead 400 times over for "
      "barely any actual work per task — the sweet spot tracks available cores and data volume, "
      "not an arbitrarily large or small fixed number."
)

     1 partitions: 0.164s


    16 partitions: 0.249s


   400 partitions: 0.701s

Too few partitions (1) can't use more than 1 core for this stage. Too many partitions (400, for only 50,000 rows) pays fixed per-task scheduling overhead 400 times over for barely any actual work per task — the sweet spot tracks available cores and data volume, not an arbitrarily large or small fixed number.


## Summary table

In [18]:
print(f"{'Operation':30s} {'multiprocessing':>18s} {'PySpark local[*]':>18s}")
print(f"{'groupBy(category).agg()':30s} {t_mp_groupby:17.3f}s {t_spark_groupby:17.3f}s")
print(f"{'join(customers)':30s} {t_mp_join:17.3f}s {t_spark_join:17.3f}s")

Operation                         multiprocessing   PySpark local[*]
groupBy(category).agg()                    0.452s             0.174s
join(customers)                            0.531s             0.164s


In [19]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
